# Noisy-label robustness: Full FT vs LoRA vs DoRA (Qwen3-0.6B)

1. Upload the whole `NLP Final` folder to Google Drive (My Drive).
2. Runtime → Change runtime type → **GPU** (L4 or A100 strongly preferred; T4 works but is slower and uses fp16).
3. Run the cells top to bottom. Results are written to Drive, so a disconnect loses nothing — just re-run.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
%cd "/content/drive/MyDrive/NLP Final"
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
!pip install -q -r requirements.txt

## 1. Check the data and the corruption

In [ ]:
!python src/data.py --n_train 20000 --noise 0.3

## 2. One quick pilot run (~5–15 min) to check speed and memory
If full FT runs out of memory, add `--grad_checkpointing` or `--batch_size 8 --grad_accum 2`.

In [ ]:
!python src/train.py --method lora --noise 0.3 --seed 1 --epochs 1 --n_train 20000

## 3. Full grid (3 methods × 2 noise levels × seeds)
Start with one seed; add seeds 2 and 3 once it works. Re-running skips finished runs.

**Delete `runs/lora_noise0.30_seed1` first** if it holds the 1-epoch pilot, otherwise it will be skipped.

In [ ]:
!rm -rf runs/lora_noise0.30_seed1
!SEEDS="1" bash scripts/run_all.sh

In [ ]:
!SEEDS="2 3" bash scripts/run_all.sh

## 4. Results tables and figures

In [ ]:
!python src/analyze.py
from IPython.display import Image, Markdown, display
display(Markdown(open('analysis/summary.md').read()))
for f in ['fig_final_bars', 'fig_memorization', 'fig_sst2_curve']:
    display(Image(f'analysis/{f}.png'))

## 5. Build the error-analysis sheets
Then download the CSVs in `analysis/errors/` and label the `category` column by hand (see `CATEGORIES.md`).

In [ ]:
!python src/error_analysis.py sample
display(Markdown(open('analysis/errors/auto_breakdown.md').read()))

In [ ]:
# after labeling: count categories per method
!python src/error_analysis.py tally analysis/errors/sst2_errors_noise0.30.csv